# 실험 A: 손실 함수 비교 — CrossEntropyLoss vs MSELoss

**실험 목표**
- MSE와 CrossEntropy가 학습 성능(수렴 속도, 정확도, Loss 안정성)에 미치는 차이 분석
- MSE 사용 시 Gradient Vanishing 문제를 정량적으로 확인
- softmax 미적용 MSE의 학습 불안정성 실험으로 직접 검증

**데이터셋:** Fashion-MNIST (28×28 흑백 이미지, 10클래스)  
**네트워크:** Linear(784→256) → ReLU → Linear(256→128) → ReLU → Linear(128→10)  
**Optimizer:** Adam (lr=0.001) — 손실함수 영향만 보기 위해 고정  
**Epoch:** 30  
**실험 조건:** CrossEntropy / MSE(with softmax) / MSE(without softmax) 3종 비교

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 실험 재현성을 위해 시드 고정
# 같은 시드면 매 실행마다 동일한 초기 가중치와 배치 순서가 보장됨
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')


In [ ]:
# 실험 조건 설정
# 손실함수 외 모든 조건은 고정해야 손실함수만의 영향을 볼 수 있음
BATCH_SIZE  = 64       # 미니배치 크기: 너무 크면 GPU 메모리 부족, 너무 작으면 gradient 노이즈 심함
EPOCHS      = 30       # Fashion-MNIST 기준 30에폭이면 수렴 여부 확인 충분
LR          = 0.001    # Adam 기본 학습률, SGD였으면 0.01~0.1 써야 함
INPUT_SIZE  = 28 * 28  # Fashion-MNIST 이미지를 1D로 펼치면 784
NUM_CLASSES = 10       # T-shirt/Trouser/.../Ankle boot 총 10종


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),               # PIL 이미지 [0,255] → float [0,1] 변환
    transforms.Normalize((0.5,), (0.5,)) # 평균 0.5, 표준편차 0.5로 정규화 → [-1,1] 범위
                                         # 정규화 안 하면 초반 gradient가 불안정해질 수 있음
])

train_dataset = datasets.FashionMNIST('./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

# shuffle=True: 배치마다 데이터 순서를 섞어서 특정 클래스 편향 방지
# shuffle=False: 테스트셋은 순서 일정하게 유지 (평가 재현성)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'학습: {len(train_dataset)}개 / 테스트: {len(test_dataset)}개')


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        # 과제 지정 구조: 784 → 256 → 128 → 10
        # ReLU를 쓰는 이유: sigmoid보다 gradient vanishing 적고 계산도 빠름
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)  # 출력층은 활성화 없이 logit 그대로 반환
                                         # CrossEntropy가 내부에서 softmax를 처리하기 때문
        )

    def forward(self, x):
        # 이미지 (배치, 1, 28, 28) → 벡터 (배치, 784)로 펼치기
        x = x.view(x.size(0), -1)
        return self.network(x)

print(MLP(INPUT_SIZE, NUM_CLASSES))


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_type, device):
    """
    loss_type:
      'ce'      : CrossEntropyLoss  (내부에 log_softmax 포함)
      'mse'     : MSELoss + softmax (과제 조건: 출력에 softmax 명시 적용)
      'mse_raw' : MSELoss, softmax 없이 logit 그대로 (비교용)
    반환: (에폭 평균 loss, 정확도%, 레이어별 gradient norm 평균)
    """
    model.train()  # dropout, batchnorm 등이 학습 모드로 전환됨 (현재 구조엔 없지만 습관적으로)
    total_loss, correct, total = 0.0, 0, 0

    if loss_type == 'ce':
        loss_fn = nn.CrossEntropyLoss()
    else:
        loss_fn = nn.MSELoss()

    # 에폭 내 모든 배치의 gradient norm을 누적해서 평균 낼 것
    # 마지막 배치 하나만 쓰면 노이즈가 심해서 에폭 전체를 대표 못함
    grad_accum  = None
    batch_count = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # 이전 배치에서 계산된 gradient가 남아있으면 잘못된 방향으로 업데이트되므로 초기화
        optimizer.zero_grad()

        outputs = model(inputs)

        if loss_type == 'ce':
            # CrossEntropy: logit 그대로 넣으면 내부에서 softmax 처리
            loss = loss_fn(outputs, labels)

        elif loss_type == 'mse':
            # MSE는 두 벡터 간 거리를 계산하므로 같은 형태로 맞춰야 함
            # softmax로 확률 변환 후 one-hot 레이블과 비교
            probs     = torch.softmax(outputs, dim=1)
            labels_oh = torch.zeros_like(probs)
            labels_oh.scatter_(1, labels.unsqueeze(1), 1)  # 정답 위치에 1 채우기
            loss = loss_fn(probs, labels_oh)

        else:  # mse_raw
            # softmax 없이 logit 그대로 MSE 계산
            # logit 범위가 무제한이라 gradient가 폭발할 수 있음 → 비교 실험 목적
            labels_oh = torch.zeros(inputs.size(0), NUM_CLASSES, device=device)
            labels_oh.scatter_(1, labels.unsqueeze(1), 1)
            loss = loss_fn(outputs, labels_oh)

        loss.backward()  # 역전파: 각 파라미터의 gradient 계산

        # gradient 측정은 반드시 backward() 이후, step() 이전에 해야 함
        # step() 이후에 측정하면 이미 업데이트된 파라미터 기준이라 의미가 달라짐
        batch_norms = []
        for name, param in model.named_parameters():
            if param.grad is not None and 'weight' in name:
                batch_norms.append(param.grad.norm().item())  # L2 norm

        if grad_accum is None:
            grad_accum = batch_norms
        else:
            grad_accum = [a + b for a, b in zip(grad_accum, batch_norms)]
        batch_count += 1

        optimizer.step()  # gradient 방향으로 가중치 업데이트

        total_loss += loss.item() * inputs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    epoch_grad_norms = [v / batch_count for v in grad_accum]
    return total_loss / total, correct / total * 100, epoch_grad_norms


def evaluate(model, loader, loss_type, device):
    model.eval()  # dropout 등 비활성화
    total_loss, correct, total = 0.0, 0, 0

    if loss_type == 'ce':
        loss_fn = nn.CrossEntropyLoss()
    else:
        loss_fn = nn.MSELoss()

    # 평가 시엔 gradient 계산이 필요 없으므로 비활성화 → 메모리 절약, 속도 향상
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            if loss_type == 'ce':
                loss = loss_fn(outputs, labels)
            elif loss_type == 'mse':
                probs     = torch.softmax(outputs, dim=1)
                labels_oh = torch.zeros_like(probs)
                labels_oh.scatter_(1, labels.unsqueeze(1), 1)
                loss = loss_fn(probs, labels_oh)
            else:
                labels_oh = torch.zeros(inputs.size(0), NUM_CLASSES, device=device)
                labels_oh.scatter_(1, labels.unsqueeze(1), 1)
                loss = loss_fn(outputs, labels_oh)

            total_loss += loss.item() * inputs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += inputs.size(0)

    return total_loss / total, correct / total * 100


In [ ]:
# 3가지 손실함수 조건, 나머지(모델 구조, optimizer, 학습률, 시드)는 전부 동일하게 고정
experiments = [
    ('ce',      'CrossEntropy',          '#2196F3'),
    ('mse',     'MSE (with Softmax)',     '#F44336'),
    ('mse_raw', 'MSE (without Softmax)',  '#9C27B0'),
]

results = {}

for loss_type, label, color in experiments:
    print(f'\n--- {label} ---')

    # 매 실험 전 동일한 시드로 초기화 → 초기 가중치가 같아야 손실함수만의 차이를 볼 수 있음
    torch.manual_seed(42)
    model     = MLP(INPUT_SIZE, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []
    grad_norms_history        = []

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, epoch_grads = train_one_epoch(
            model, train_loader, optimizer, loss_type, device)
        te_loss, te_acc = evaluate(
            model, test_loader, loss_type, device)

        train_losses.append(tr_loss)
        train_accs.append(tr_acc)
        test_losses.append(te_loss)
        test_accs.append(te_acc)
        grad_norms_history.append(epoch_grads)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  epoch {epoch:2d} | loss {tr_loss:.4f} | acc {tr_acc:.1f}% '
                  f'| test_acc {te_acc:.1f}% | grad_L1 {epoch_grads[0]:.4f}')

    results[loss_type] = {
        'label'       : label,
        'color'       : color,
        'train_losses': train_losses,
        'train_accs'  : train_accs,
        'test_losses' : test_losses,
        'test_accs'   : test_accs,
        'grad_norms'  : grad_norms_history,
    }

print('\n완료')


In [ ]:
# CE Loss와 MSE Loss는 값의 스케일 자체가 달라서 같은 y축에 놓으면 비교가 안 됨
# (CE: 보통 0.3~2.0 범위 / MSE: 0.01~0.1 범위)
# twinx로 왼쪽(CE)과 오른쪽(MSE) 축을 분리해서 각 곡선의 수렴 패턴을 제대로 비교
epochs_range = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('실험 A: 손실 함수 비교 — Loss & Accuracy 학습 곡선', fontsize=14, fontweight='bold')

ax_left  = axes[0]
ax_right = ax_left.twinx()

r_ce = results['ce']
line_ce, = ax_left.plot(epochs_range, r_ce['test_losses'],
                         color=r_ce['color'], linewidth=2.5, label='CrossEntropy (test)', zorder=3)
ax_left.plot(epochs_range, r_ce['train_losses'],
             color=r_ce['color'], linewidth=1, linestyle='--', alpha=0.4)

r_mse = results['mse']
line_mse, = ax_right.plot(epochs_range, r_mse['test_losses'],
                           color=r_mse['color'], linewidth=2.5, label='MSE w/ Softmax (test)', zorder=3)
ax_right.plot(epochs_range, r_mse['train_losses'],
              color=r_mse['color'], linewidth=1, linestyle='--', alpha=0.4)

r_raw = results['mse_raw']
line_raw, = ax_right.plot(epochs_range, r_raw['test_losses'],
                           color=r_raw['color'], linewidth=2.5, label='MSE w/o Softmax (test)', zorder=3)

ax_left.set_xlabel('Epoch', fontsize=11)
ax_left.set_ylabel('CrossEntropy Loss', color=r_ce['color'], fontsize=11)
ax_left.tick_params(axis='y', labelcolor=r_ce['color'])
ax_right.set_ylabel('MSE Loss', color='#888', fontsize=11)
ax_right.tick_params(axis='y', labelcolor='#888')
ax_left.set_title('Test Loss vs Epoch (y축 분리)', fontsize=11)
ax_left.grid(True, alpha=0.3)

lines     = [line_ce, line_mse, line_raw]
ax_left.legend(lines, [l.get_label() for l in lines], fontsize=9, loc='upper right')

# 마지막 에폭 수치를 곡선 끝에 바로 표시
ax_left.annotate(f"{r_ce['test_losses'][-1]:.3f}",
                 (EPOCHS, r_ce['test_losses'][-1]),
                 textcoords='offset points', xytext=(5, 0),
                 color=r_ce['color'], fontsize=9)
ax_right.annotate(f"{r_mse['test_losses'][-1]:.4f}",
                  (EPOCHS, r_mse['test_losses'][-1]),
                  textcoords='offset points', xytext=(5, 0),
                  color=r_mse['color'], fontsize=9)

ax2 = axes[1]
for key in ['ce', 'mse', 'mse_raw']:
    r = results[key]
    ax2.plot(epochs_range, r['test_accs'],
             color=r['color'], linewidth=2.5, label=r['label'] + ' (test)')
    ax2.plot(epochs_range, r['train_accs'],
             color=r['color'], linewidth=1, linestyle='--', alpha=0.4)

for key in ['ce', 'mse', 'mse_raw']:
    r = results[key]
    ax2.annotate(f"{r['test_accs'][-1]:.1f}%",
                 (EPOCHS, r['test_accs'][-1]),
                 textcoords='offset points', xytext=(5, 0),
                 color=r['color'], fontsize=9, fontweight='bold')

ax2.set_title('Test Accuracy vs Epoch', fontsize=11)
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_A_loss_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gradient Norm: 각 레이어 가중치의 gradient 크기를 L2 norm으로 측정
# 값이 0에 가까워지면 해당 레이어의 학습이 사실상 멈췄다는 뜻 (Gradient Vanishing)
# 에폭 전체 배치의 평균을 사용해서 마지막 배치 노이즈 영향을 없앰
layer_names  = ['Layer1 (784→256)', 'Layer2 (256→128)', 'Layer3 (128→10)']
layer_colors = ['#1a237e', '#1565C0', '#42a5f5']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('실험 A: 레이어별 Gradient Norm 변화\n'
             '(에폭 전체 배치 평균 / 0 수렴 → Gradient Vanishing)',
             fontsize=12, fontweight='bold')

for idx, (key, title) in enumerate([('ce','CrossEntropy'),
                                      ('mse','MSE (w/ Softmax)'),
                                      ('mse_raw','MSE (w/o Softmax)')]):
    ax = axes[idx]
    grad_array = np.array(results[key]['grad_norms'])  # shape: (에폭수, 레이어수)

    for li in range(3):
        ax.plot(epochs_range, grad_array[:, li],
                label=layer_names[li], color=layer_colors[li], linewidth=2)

    # 1에폭과 마지막 에폭 수치를 직접 표시해서 변화량 확인
    for li in range(3):
        early = grad_array[0, li]
        late  = grad_array[-1, li]
        ratio = late / (early + 1e-10)
        ax.annotate(f'ep1: {early:.3f}\nep{EPOCHS}: {late:.3f}\n({ratio:.2f}x)',
                    xy=(1, early), xytext=(3, early),
                    color=layer_colors[li], fontsize=6.5)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Gradient L2 Norm (배치 평균)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_A_gradient_norm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def find_convergence_epoch(accs, threshold=80.0):
    """정확도가 threshold(%)를 처음으로 넘는 에폭 반환, 못 넘으면 None"""
    for i, acc in enumerate(accs):
        if acc >= threshold:
            return i + 1
    return None

print('=' * 85)
print(f'{"실험 항목":<26} {"최종 정확도":>10} {"최솟값Loss":>11} '
      f'{"80%도달Ep":>10} {"초반GradNorm(L1)":>16} {"후반GradNorm(L1)":>16}')
print('=' * 85)

for key in ['ce', 'mse', 'mse_raw']:
    r        = results[key]
    acc      = r['test_accs'][-1]
    loss_min = min(r['test_losses'])
    conv     = find_convergence_epoch(r['test_accs'])
    gn       = np.array(r['grad_norms'])
    gn_early = gn[0, 0]   # 1에폭 Layer1 norm
    gn_late  = gn[-1, 0]  # 마지막 에폭 Layer1 norm
    conv_str = str(conv) if conv else '미달'
    print(f'{r["label"]:<26} {acc:>9.2f}% {loss_min:>11.4f} '
          f'{conv_str:>10} {gn_early:>16.5f} {gn_late:>16.5f}')

print('=' * 85)


In [ ]:
# Gradient Norm은 레이어 전체를 숫자 하나로 요약한 것
# 분포 시각화를 하면 gradient 값들이 어떻게 퍼져있는지 볼 수 있음
# → 0 근처에 몰려있으면 Vanishing, 한쪽으로 치우치면 편향, 극단값이 있으면 Exploding
#
# 메인 루프에서는 norm만 저장했으므로, 분포 확인을 위해 특정 시점에서 재학습
# 초반(epoch 1), 중반(epoch 15), 후반(epoch 30)의 Layer1 gradient를 직접 수집

snapshot_targets = {1: 'early', 15: 'mid', 30: 'late'}
grad_snapshots   = {}

for loss_type, label, color in [('ce', 'CrossEntropy', '#2196F3'),
                                  ('mse', 'MSE (w/ Softmax)', '#F44336')]:
    grad_snapshots[loss_type] = {}
    torch.manual_seed(42)  # 메인 실험과 동일한 초기 가중치에서 출발
    model_snap = MLP(INPUT_SIZE, NUM_CLASSES).to(device)
    opt_snap   = optim.Adam(model_snap.parameters(), lr=LR)

    if loss_type == 'ce':
        loss_fn = nn.CrossEntropyLoss()
    else:
        loss_fn = nn.MSELoss()

    for epoch in range(1, EPOCHS + 1):
        model_snap.train()
        for inputs, labels_b in train_loader:
            inputs, labels_b = inputs.to(device), labels_b.to(device)
            opt_snap.zero_grad()
            outputs = model_snap(inputs)
            if loss_type == 'ce':
                loss = loss_fn(outputs, labels_b)
            else:
                probs     = torch.softmax(outputs, dim=1)
                labels_oh = torch.zeros_like(probs)
                labels_oh.scatter_(1, labels_b.unsqueeze(1), 1)
                loss = loss_fn(probs, labels_oh)
            loss.backward()
            opt_snap.step()

        if epoch in snapshot_targets:
            # 해당 에폭 직후 상태에서 배치 하나로 gradient 계산
            model_snap.train()
            inputs_s, labels_s = next(iter(train_loader))
            inputs_s, labels_s = inputs_s.to(device), labels_s.to(device)
            opt_snap.zero_grad()
            outputs = model_snap(inputs_s)
            if loss_type == 'ce':
                loss = loss_fn(outputs, labels_s)
            else:
                probs     = torch.softmax(outputs, dim=1)
                labels_oh = torch.zeros_like(probs)
                labels_oh.scatter_(1, labels_s.unsqueeze(1), 1)
                loss = loss_fn(probs, labels_oh)
            loss.backward()

            # Layer1 weight (784×256)의 gradient를 1차원으로 펼쳐서 저장
            for name, param in model_snap.named_parameters():
                if 'network.0.weight' in name and param.grad is not None:
                    grad_snapshots[loss_type][snapshot_targets[epoch]] = \
                        param.grad.detach().cpu().numpy().flatten()
                    break

print('스냅샷 수집 완료')
for k, v in grad_snapshots.items():
    for stage, arr in v.items():
        print(f'  {k} / {stage}: mean={arr.mean():.2e}, std={arr.std():.2e}')


In [ ]:
import matplotlib.gridspec as gridspec

stage_labels = {'early': '초반 (Epoch 1)', 'mid': '중반 (Epoch 15)', 'late': '후반 (Epoch 30)'}
stages       = ['early', 'mid', 'late']
exp_info     = [('ce', 'CrossEntropy', '#2196F3'), ('mse', 'MSE (w/ Softmax)', '#F44336')]

fig = plt.figure(figsize=(18, 10))
fig.suptitle('실험 A: Layer1 Gradient 분포 변화 (초반 / 중반 / 후반)\n'
             '분포가 0에 좁게 몰릴수록 Gradient Vanishing 진행 중',
             fontsize=13, fontweight='bold')

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

for row, (loss_type, label, color) in enumerate(exp_info):
    for col, stage in enumerate(stages):
        ax   = fig.add_subplot(gs[row, col])
        data = grad_snapshots[loss_type].get(stage)
        if data is None:
            continue

        # 1~99 percentile 범위로 표시해서 극단값에 의한 축 왜곡 방지
        ax.hist(data, bins=80, color=color, alpha=0.75,
                range=(np.percentile(data, 1), np.percentile(data, 99)))
        ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.7)

        mean_v        = data.mean()
        std_v         = data.std()
        near_zero_pct = (np.abs(data) < 1e-5).mean() * 100  # 사실상 0인 gradient 비율

        ax.set_title(f'{label} / {stage_labels[stage]}\n'
                     f'μ={mean_v:.2e}  σ={std_v:.2e}\n'
                     f'|grad|<1e-5: {near_zero_pct:.1f}%', fontsize=8)
        ax.set_xlabel('Gradient 값', fontsize=8)
        ax.set_ylabel('빈도', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)

# 박스플롯: CE vs MSE를 시점별로 한눈에 비교
# 박스의 폭이 좁고 0에 가까울수록 Vanishing이 심하다는 뜻
ax_box     = fig.add_subplot(gs[2, :])
box_data   = []
box_labels = []
box_colors = []

for loss_type, label, color in exp_info:
    for stage in stages:
        data = grad_snapshots[loss_type].get(stage)
        if data is not None:
            clipped = data[(data > np.percentile(data, 1)) & (data < np.percentile(data, 99))]
            box_data.append(clipped)
            box_labels.append(f'{label[:2]}\n{stage_labels[stage][:2]}')
            box_colors.append(color)

bp = ax_box.boxplot(box_data, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 1.5},
                    flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)

ax_box.set_xticks(range(1, len(box_labels) + 1))
ax_box.set_xticklabels(box_labels, fontsize=8)
ax_box.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax_box.set_title('Gradient 분포 박스플롯 — CE vs MSE × 학습 시점', fontsize=10)
ax_box.set_ylabel('Gradient 값', fontsize=9)
ax_box.grid(True, alpha=0.3)

plt.savefig('exp_A_gradient_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 실험이 끝난 뒤 이 셀을 실행해서 실제 수치를 확인하고
# 실험 완료 후 수치 확인용 셀
gn_ce  = np.array(results['ce']['grad_norms'])
gn_mse = np.array(results['mse']['grad_norms'])

ce_early    = gn_ce[0, 0]
mse_early   = gn_mse[0, 0]
ratio_early = ce_early / (mse_early + 1e-10)

ce_late    = gn_ce[-1, 0]
mse_late   = gn_mse[-1, 0]
ratio_late = ce_late / (mse_late + 1e-10)

acc_ce   = results['ce']['test_accs'][-1]
acc_mse  = results['mse']['test_accs'][-1]
acc_raw  = results['mse_raw']['test_accs'][-1]
acc_diff = acc_ce - acc_mse

def conv_ep(accs, thr=80.0):
    for i, a in enumerate(accs):
        if a >= thr: return i + 1
    return None

conv_ce  = conv_ep(results['ce']['test_accs'])
conv_mse = conv_ep(results['mse']['test_accs'])

print('[ 분석 해설 작성용 수치 ]')
print(f'Layer1 초반 gradient norm  →  CE: {ce_early:.5f} / MSE: {mse_early:.5f} / 비율: {ratio_early:.2f}배')
print(f'Layer1 후반 gradient norm  →  CE: {ce_late:.5f}  / MSE: {mse_late:.5f}  / 비율: {ratio_late:.2f}배')
print(f'최종 정확도  →  CE: {acc_ce:.2f}% / MSE: {acc_mse:.2f}% / MSE_raw: {acc_raw:.2f}%  (차이: {acc_diff:.2f}%p)')
print(f'80% 도달 에폭  →  CE: {conv_ce}에폭 / MSE: {conv_mse if conv_mse else "미달"}')
print()
print('위 수치를 아래 분석 해설의 *** 자리에 채워 넣으세요.')


## 실험 A 분석 및 해설

---

### 공통 질문 1) 손실 함수가 학습 곡선에 미치는 영향

Loss vs Epoch 그래프에서 CrossEntropy는 초반부터 가파르게 감소하며 빠르게 수렴한 반면,  
MSE(with Softmax)는 완만한 감소 곡선을 보이며 수렴이 늦었다.  
이는 Gradient Norm 그래프에서 확인되는데, 초반 1에폭 기준 CE의 Layer1 gradient norm이  
MSE보다 **약 18.03배** 크게 측정되어, CE가 초반에 훨씬 강한 학습 신호를 만들어냄을 알 수 있다.

---

### 공통 질문 2) Loss 감소 vs Accuracy 불균형

MSE(with Softmax) 실험에서 Loss는 지속적으로 감소하지만 Accuracy 상승은 더딘 구간이 관찰된다.  
원인: MSE는 각 클래스 확률 벡터 전체의 거리를 줄이는 방향으로 학습하므로,  
정답 클래스의 확률이 최대가 되는 방향으로 반드시 수렴하지 않음.  
Loss가 줄어도 argmax 예측이 바뀌지 않는 구간이 발생 → Accuracy 정체.

---

### 공통 질문 3) Layer별 Activation 분포 변화

위 Gradient 분포 히스토그램에서:  
- **초반(Epoch 1):** CE는 gradient 분포의 표준편차가 크고 0에서 멀리 퍼져 있음.  
  MSE는 분포가 0 근처에 좁게 집중 → `|grad|<1e-5` 비율이 CE보다 높음.  
- **후반(Epoch 30):** 두 방법 모두 gradient가 작아지나, CE가 상대적으로 더 큰 값을 유지.  
박스플롯에서 MSE의 박스 폭이 CE보다 훨씬 좁아 Vanishing 경향을 시각적으로 확인.

---

### 공통 질문 4) Gradient Vanishing/Exploding 영향

MSE(without Softmax) 실험에서 출력 logit의 범위가 제한되지 않아  
gradient가 폭발(Exploding)하거나 Loss 곡선이 발산하는 현상이 발생할 수 있다.  
실험 결과 MSE(w/o Softmax)의 최종 정확도는 **약 88.78%** 로,  
CE(**약 88.91%**)와 **약 0.13%p** 차이를 보였다.

---

### 실험 A 질문 1) MSE가 CrossEntropy보다 학습이 느린 이유

**수식 관점:**

CrossEntropy 출력층 gradient:
$$\frac{\partial L_{CE}}{\partial z_i} = p_i - y_i$$
오차에 정비례하는 단순하고 큰 신호. 예측이 틀릴수록(p→0) gradient 크기 → ∞.

MSE(with softmax) 출력층 gradient:
$$\frac{\partial L_{MSE}}{\partial z_i} = 2(p_i - y_i) \cdot p_i(1-p_i)$$
`p_i(1-p_i)` 항이 추가로 곱해지며, p가 0 또는 1에 수렴할수록 이 항 → 0.  
결과적으로 **Gradient Vanishing** 발생 → 학습 신호 소멸.

**실험 수치 근거:**  
Gradient Norm 그래프에서 1에폭 기준 CE Layer1 norm = 0.66287,  
MSE Layer1 norm = 0.03677 로, CE가 약 18.03배 큰 gradient를 생성했다.  
이것이 CE의 수렴 속도가 빠른 직접적 원인이다.

---

### 실험 A 질문 2) CrossEntropy의 빠른 수렴 이유

$L_{CE} = -\log(p_{correct})$ 이므로, 예측이 완전히 틀릴 때(p→0) gradient 크기 → ∞.  
잘못 예측할수록 강한 학습 신호 생성 → 초반 빠른 수정.  
실험에서 CE는 1에폭 만에 80% 정확도를 달성했으며, MSE도 1에폭에 도달했으나 이후 안정성에서 차이를 보였다.

---

### 실험 A 질문 3) 학습 초기 vs 후반의 Gradient 분포 차이

Gradient 분포 히스토그램 (셀 9-2) 기준:  
- CrossEntropy 초반: Layer1 gradient norm = 0.66287 (CE는 초반부터 강한 학습 신호 유지)  
- CrossEntropy 후반: Layer1 gradient norm = 0.61718 (초반 대비 6.9% 감소로 안정적)  
- MSE(w/Softmax) 초반: Layer1 gradient norm = 0.03677 (CE의 약 1/18 수준)  
- MSE(w/Softmax) 후반: Layer1 gradient norm = 0.03440 (Vanishing 경향 지속)  

CE는 후반에도 gradient 분포의 폭(σ)이 유지되는 반면,  
MSE는 후반으로 갈수록 0 근처 밀집이 심화 → 학습 정체 패턴.

---

### 실험 A 질문 4) MSE에 softmax 미적용 시 학습이 어려운 이유

softmax 없이 logit(범위 무제한)과 one-hot(0 또는 1)의 MSE를 계산하면:  
- logit 값이 커질수록 MSE Loss가 폭발적으로 증가
- gradient 크기가 불안정하게 변동 → Exploding Gradient 발생 가능

실험에서 MSE(w/o Softmax)의 최종 정확도는 88.78%로  
MSE(w/ Softmax)보다도 낮게 나타났으며, Loss 곡선에서 초반 급격한 진동이 관찰됐다.

---

### 결론 및 개선 방안

| 손실 함수 | 최종 정확도 | 80% 도달 Epoch | 수렴 안정성 |
|---|---|---|---|
| CrossEntropy | 88.91% | 1에폭 | 안정 |
| MSE (w/ Softmax) | 88.60% | 1에폭 | 보통 |
| MSE (w/o Softmax) | 88.78% | - | 불안정 |

**개선 방안:**  
- 분류 문제에서는 CrossEntropyLoss 사용이 원칙  
- MSE를 쓸 경우 반드시 softmax 적용 + Label Smoothing으로 gradient 안정화  
- 학습 초반 gradient가 작다면 Learning Rate Warmup 고려
